# Training Log Visualisation

This notebook provides a tool to visualize training logs saved as JSON files. You can select a metric (validation metrics or loss) to plot on the y-axis, with epochs on the x-axis. Each seed run is shown as a separate line with its own color.

In [1]:
import json
import os
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from ipywidgets import interact, Dropdown

# Utility to load log JSON

def load_log_json(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

def extract_metrics(log_data):
    # Returns dict: seed -> {epoch_history: [...], summary: {...}}
    seed_runs = log_data.get('seed_runs', [])
    metrics = {}
    for run in seed_runs:
        seed = run.get('seed', None)
        if seed is not None:
            metrics[seed] = run.get('epoch_history', [])
    return metrics

def get_metric_names(metrics_by_seed):
    # Find all possible metric keys in val_metrics and test_metrics
    metric_names = set(['train_loss'])
    for seed, epochs in metrics_by_seed.items():
        for entry in epochs:
            metric_names.add('train_loss')
            if 'val_metrics' in entry:
                metric_names.update(entry['val_metrics'].keys())
            if 'test_metrics' in entry and entry['test_metrics']:
                metric_names.update(entry['test_metrics'].keys())
    return sorted(metric_names)

def plot_metrics(metrics_by_seed, metric_name):
    plt.figure(figsize=(8,5))
    cmap = get_cmap('tab10')
    for idx, (seed, epochs) in enumerate(metrics_by_seed.items()):
        xs = [e['epoch'] for e in epochs]
        if metric_name == 'train_loss':
            ys = [e['train_loss'] for e in epochs]
        else:
            ys = [e.get('val_metrics', {}).get(metric_name, None) for e in epochs]
        plt.plot(xs, ys, label=f'Seed {seed}', color=cmap(idx))
    plt.xlabel('Epoch')
    plt.ylabel(metric_name)
    plt.title(f'{metric_name} over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

def visualize_log(json_path):
    log_data = load_log_json(json_path)
    metrics_by_seed = extract_metrics(log_data)
    metric_names = get_metric_names(metrics_by_seed)
    interact(lambda metric: plot_metrics(metrics_by_seed, metric),
             metric=Dropdown(options=metric_names, value=metric_names[0], description='Metric:'))



In [2]:
def plot_mean_val_mae(json_path):
    """
    Plots the mean validation MAE over 5 seeds per epoch from a log JSON config.
    y-axis: mean validation_mae, x-axis: epoch number
    """
    import json
    import matplotlib.pyplot as plt
    
    # Load log data
    with open(json_path, 'r') as f:
        log_data = json.load(f)
    
    # Extract per-seed epoch histories
    seed_runs = log_data.get('seed_runs', [])
    if not seed_runs:
        print('No seed_runs found in log.')
        return
    
    # Build epoch -> list of val_mae for all seeds
    epoch_mae = {}
    for run in seed_runs:
        for entry in run.get('epoch_history', []):
            epoch = entry['epoch']
            val_mae = entry.get('val_metrics', {}).get('mae', None)
            if val_mae is not None:
                epoch_mae.setdefault(epoch, []).append(val_mae)
    
    # Calculate mean MAE per epoch
    epochs = sorted(epoch_mae.keys())
    mean_mae = [sum(epoch_mae[e])/len(epoch_mae[e]) for e in epochs]
    
    # Plot
    plt.figure(figsize=(8,5))
    plt.plot(epochs, mean_mae, marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Validation MAE (5 seeds)')
    plt.title('Mean Validation MAE over Epochs (5 seeds)')
    plt.grid(True)
    plt.show()

In [3]:
import os
import re

def get_available_metrics(folder_path, metric_group='test_metrics'):
    """
    Collect all metric names available in epoch_history[metric_group] across cfg_*.json files.
    """
    import json
    metric_names = set()
    config_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and f.startswith('cfg_')]
    for cfg in config_files:
        json_path = os.path.join(folder_path, cfg)
        with open(json_path, 'r') as f:
            log_data = json.load(f)
        for run in log_data.get('seed_runs', []):
            for entry in run.get('epoch_history', []):
                metric_names.update(entry.get(metric_group, {}).keys())
    return sorted(metric_names)

def plot_all_configs_metric(
    folder_path,
    metric_name='mae',
    metric_group='test_metrics',
    filter_mode=None,
    benchmark_cfg='cfg_1.json'
 ):
    """
    Plot mean metric per epoch over seeds for all cfg_*.json files in a folder.

    - metric_name: metric key from epoch_history[metric_group] (for example 'mae', 'mse', 'r2')
    - metric_group: usually 'test_metrics' (dataset-specific metrics), can also be 'val_metrics'
    - filter_mode: None (all), 'bridge' (bridge only), 'hub' (hub only), 'both' (both True)
    - benchmark_cfg is always plotted (if it contains the selected metric).
    """
    import json
    import matplotlib.pyplot as plt

    def extract_number(filename):
        match = re.search(r'(\d+)', filename)
        return int(match.group(1)) if match else float('inf')

    config_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and f.startswith('cfg_')]
    config_files.sort(key=extract_number)
    if not config_files:
        print('No cfg_*.json files found in folder.')
        return None

    available_metrics = get_available_metrics(folder_path, metric_group=metric_group)
    if metric_name not in available_metrics:
        print(f"Metric '{metric_name}' not found in {metric_group}. Available: {available_metrics}")
        return None

    plt.figure(figsize=(10, 6))

    def compute_epoch_means(log_data):
        epoch_values = {}
        for run in log_data.get('seed_runs', []):
            for entry in run.get('epoch_history', []):
                epoch = entry.get('epoch')
                metric_val = entry.get(metric_group, {}).get(metric_name, None)
                if epoch is not None and metric_val is not None:
                    epoch_values.setdefault(epoch, []).append(metric_val)
        if not epoch_values:
            return None, None
        epochs = sorted(epoch_values.keys())
        mean_values = [sum(epoch_values[e]) / len(epoch_values[e]) for e in epochs]
        return epochs, mean_values

    # Always plot benchmark first if present.
    if benchmark_cfg in config_files:
        bench_path = os.path.join(folder_path, benchmark_cfg)
        with open(bench_path, 'r') as f:
            bench_data = json.load(f)
        run_params = bench_data.get('run_params', {})
        epochs, mean_values = compute_epoch_means(bench_data)
        if epochs is not None:
            legend_label = (
                f"BENCHMARK: bridge_proc={run_params.get('process_bridge')}, "
                f"bridge_strat={run_params.get('bridge_strategy')}, "
                f"hub_proc={run_params.get('process_hub')}, "
                f"hub_strat={run_params.get('hub_strategy')}"
            )
            plt.plot(epochs, mean_values, marker='o', label=legend_label, linewidth=3, color='black')
        config_files = [f for f in config_files if f != benchmark_cfg]

    for cfg in config_files:
        json_path = os.path.join(folder_path, cfg)
        with open(json_path, 'r') as f:
            log_data = json.load(f)

        run_params = log_data.get('run_params', {})
        process_bridge = run_params.get('process_bridge', None)
        process_hub = run_params.get('process_hub', None)

        if filter_mode == 'bridge' and not (process_bridge and not process_hub):
            continue
        if filter_mode == 'hub' and not (process_hub and not process_bridge):
            continue
        if filter_mode == 'both' and not (process_bridge and process_hub):
            continue

        epochs, mean_values = compute_epoch_means(log_data)
        if epochs is None:
            continue

        legend_label = (
            f"bridge_proc={process_bridge}, "
            f"bridge_strat={run_params.get('bridge_strategy')}, "
            f"hub_proc={process_hub}, "
            f"hub_strat={run_params.get('hub_strategy')}"
        )
        plt.plot(epochs, mean_values, marker='o', label=legend_label)

    plt.xlabel('Epoch')
    plt.ylabel(f'Mean {metric_name} over seeds ({metric_group})')
    folder_name = os.path.basename(os.path.normpath(folder_path))
    parts = folder_name.split('_')
    if len(parts) >= 2:
        dataset_name = parts[0]
        task_name = parts[1]
        plot_title = f'Dataset: {dataset_name}    Task: {task_name}'
    else:
        plot_title = folder_name
    plt.title(plot_title)
    plt.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.14),
        fontsize='small'
    )
    plt.grid(True)
    fig = plt.gcf()
    plt.show()
    return fig

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

def choose_dbgnn_16combos_path(
    base_log_dir='/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/',
    default_filter_mode='both',
    default_metric_group='test_metrics'
 ):
    """
    Interactive picker for dbgnn_16combos folders with cfg_*.json files.
    Lets you choose folder, metric_group, metric_name and filter_mode before plotting.
    """
    base = Path(base_log_dir)
    if not base.exists():
        print(f'Base directory not found: {base}')
        return None

    def has_cfg_json(folder: Path) -> bool:
        return any(folder.glob('cfg_*.json'))

    def prioritize_roc_auc(metrics):
        if 'roc_auc' in metrics:
            return ['roc_auc'] + [m for m in metrics if m != 'roc_auc']
        return metrics

    def get_dataset_task_name(folder_path):
        folder_name = Path(folder_path).name
        parts = folder_name.split('_')
        if len(parts) >= 2:
            return f'{parts[0]}_{parts[1]}'
        return folder_name

    def get_png_name(folder_path, filter_mode):
        filter_name = filter_mode or 'all'
        return f"{get_dataset_task_name(folder_path)}_{filter_name}.png"

    def get_plots_dir():
        return base.resolve().parents[1] / 'plots'

    candidates = []
    for p1 in base.iterdir():
        if not p1.is_dir():
            continue
        if has_cfg_json(p1):
            candidates.append(p1)
        for p2 in p1.iterdir():
            if p2.is_dir() and has_cfg_json(p2):
                candidates.append(p2)

    sorted_candidates = sorted(candidates, key=lambda p: str(p))
    folder_options = [(p.name, str(p)) for p in sorted_candidates]
    if not folder_options:
        print(f'No folders with cfg_*.json found in: {base}')
        return None

    folder_dropdown = widgets.Dropdown(
        options=folder_options,
        value=folder_options[0][1],
        description='Folder:',
        layout=widgets.Layout(width='95%')
    )

    metric_group_dropdown = widgets.Dropdown(
        options=['test_metrics', 'val_metrics'],
        value=default_metric_group,
        description='Group:'
    )

    filter_dropdown = widgets.Dropdown(
        options=[('all', 'all'), ('bridge', 'bridge'), ('hub', 'hub'), ('both', 'both')],
        value=default_filter_mode,
        description='Filter:'
    )

    metric_dropdown = widgets.Dropdown(
        options=[],
        description='Metric:'
    )

    png_name_input = widgets.Text(
        value=get_png_name(folder_options[0][1], default_filter_mode),
        description='PNG file:',
        layout=widgets.Layout(width='60%')
    )

    button = widgets.Button(description='Plot Selected', button_style='primary')
    save_button = widgets.Button(description='Save PNG', button_style='success')
    output = widgets.Output()
    last_fig = None

    def refresh_metric_options(*_):
        selected_folder = folder_dropdown.value
        selected_group = metric_group_dropdown.value
        metrics = get_available_metrics(selected_folder, metric_group=selected_group)
        metrics = prioritize_roc_auc(metrics)
        metric_dropdown.options = metrics
        if metrics:
            metric_dropdown.value = metrics[0]

    def update_default_png_name(*_):
        png_name_input.value = get_png_name(folder_dropdown.value, filter_dropdown.value)

    def on_click(_):
        nonlocal last_fig
        with output:
            clear_output(wait=True)
            selected_folder = folder_dropdown.value
            selected_group = metric_group_dropdown.value
            selected_metric = metric_dropdown.value
            selected_filter = filter_dropdown.value
            print(f'Using: {selected_folder}')
            print(f'Plotting: {selected_group}.{selected_metric} | filter={selected_filter}')
            last_fig = plot_all_configs_metric(
                selected_folder,
                metric_name=selected_metric,
                metric_group=selected_group,
                filter_mode=selected_filter
            )

    def on_save_click(_):
        nonlocal last_fig
        with output:
            import matplotlib.pyplot as plt

            fig_to_save = last_fig
            if fig_to_save is None:
                figure_numbers = plt.get_fignums()
                if figure_numbers:
                    fig_to_save = plt.figure(figure_numbers[-1])

            if fig_to_save is None:
                print('No figure to save yet. Click Plot Selected first.')
                return

            filename = png_name_input.value.strip() or get_png_name(folder_dropdown.value, filter_dropdown.value)
            filename = Path(filename).name
            if not filename.lower().endswith('.png'):
                filename += '.png'

            plots_dir = get_plots_dir()
            plots_dir.mkdir(parents=True, exist_ok=True)
            save_path = plots_dir / filename
            fig_to_save.savefig(save_path, dpi=300, bbox_inches='tight')
            last_fig = fig_to_save
            print(f'Saved PNG: {save_path}')

    folder_dropdown.observe(refresh_metric_options, names='value')
    folder_dropdown.observe(update_default_png_name, names='value')
    metric_group_dropdown.observe(refresh_metric_options, names='value')
    filter_dropdown.observe(update_default_png_name, names='value')
    refresh_metric_options()

    button.on_click(on_click)
    save_button.on_click(on_save_click)
    display(widgets.VBox([
        folder_dropdown,
        widgets.HBox([metric_group_dropdown, metric_dropdown, filter_dropdown]),
        widgets.HBox([button, save_button]),
        png_name_input,
        output
    ]))
    return folder_dropdown, metric_group_dropdown, metric_dropdown, filter_dropdown

choose_dbgnn_16combos_path()

(Dropdown(description='Folder:', layout=Layout(width='95%'), options=(('ctu-adventureworks_adventureworks-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-adventureworks_adventureworks-original/ctu-adventureworks_adventureworks-original_sage_edge_attr'), ('ctu-geneea_geneea-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-geneea_geneea-original/ctu-geneea_geneea-original_sage_edge_attr'), ('ctu-lahman_lahman-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-lahman_lahman-original/ctu-lahman_lahman-original_sage_edge_attr'), ('ctu-northwind_northwind-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-northwind_northwind-original/ctu-northwind_northwind-original_sage_edge_attr'), ('rel-amazon_item-churn_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/rel-amazon_item-churn/rel-amazon_item-churn_sage_edge_attr'), ('rel-amazon_item-ltv_sage_edge_attr', '/hom

In [5]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_single_config_val_test_metric(json_path, metric_name):
    """
    Plot mean train_loss, validation metric, and test metric per epoch over seeds
    for one cfg_*.json file.
    """
    import json
    import matplotlib.pyplot as plt

    with open(json_path, 'r') as f:
        log_data = json.load(f)

    def compute_epoch_means(metric_group=None, metric_key=None, direct_key=None):
        epoch_values = {}
        for run in log_data.get('seed_runs', []):
            for entry in run.get('epoch_history', []):
                epoch = entry.get('epoch')
                if direct_key is not None:
                    metric_val = entry.get(direct_key, None)
                else:
                    metric_val = entry.get(metric_group, {}).get(metric_key, None)
                if epoch is not None and metric_val is not None:
                    epoch_values.setdefault(epoch, []).append(metric_val)
        if not epoch_values:
            return None, None
        epochs = sorted(epoch_values.keys())
        mean_values = [sum(epoch_values[e]) / len(epoch_values[e]) for e in epochs]
        return epochs, mean_values

    loss_epochs, loss_means = compute_epoch_means(direct_key='train_loss')
    val_epochs, val_means = compute_epoch_means(metric_group='val_metrics', metric_key=metric_name)
    test_epochs, test_means = compute_epoch_means(metric_group='test_metrics', metric_key=metric_name)

    if loss_epochs is None and val_epochs is None and test_epochs is None:
        print(f"No train_loss, val_metrics['{metric_name}'], or test_metrics['{metric_name}'] found for {json_path}")
        return None

    plt.figure(figsize=(10, 6))

    if loss_epochs is not None:
        plt.plot(loss_epochs, loss_means, marker='o', label='training loss')
    if val_epochs is not None:
        plt.plot(val_epochs, val_means, marker='o', label=f'validation {metric_name}')
    if test_epochs is not None:
        plt.plot(test_epochs, test_means, marker='o', label=f'test {metric_name}')

    plt.xlabel('Epoch')
    plt.ylabel('Mean value over seeds')
    plt.title(f'{Path(json_path).name}: train_loss and {metric_name}')
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), fontsize='small')
    plt.grid(True)
    fig = plt.gcf()
    plt.show()
    return fig

def choose_single_config_metric_plot(
    base_log_dir='/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/'
 ):
    """
    Interactive picker for choosing folder, config and metric, then plotting
    train_loss together with validation and test curves for the selected metric.
    The config dropdown shows bridge/hub processing and strategy settings.
    """
    base = Path(base_log_dir)
    if not base.exists():
        print(f'Base directory not found: {base}')
        return None

    def has_cfg_json(folder: Path) -> bool:
        return any(folder.glob('cfg_*.json'))

    def prioritize_roc_auc(metrics):
        if 'roc_auc' in metrics:
            return ['roc_auc'] + [m for m in metrics if m != 'roc_auc']
        return metrics

    def get_dataset_task_name(folder_path):
        folder_name = Path(folder_path).name
        parts = folder_name.split('_')
        if len(parts) >= 2:
            return f'{parts[0]}_{parts[1]}'
        return folder_name

    def get_plots_dir():
        return base.resolve().parents[1] / 'plots'

    def sorted_cfg_files(folder_path):
        import re
        def extract_number(filename):
            match = re.search(r'(\d+)', filename)
            return int(match.group(1)) if match else float('inf')
        config_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and f.startswith('cfg_')]
        return sorted(config_files, key=extract_number)

    def get_metric_names_for_config(json_path):
        import json
        metric_names = set()
        with open(json_path, 'r') as f:
            log_data = json.load(f)
        for run in log_data.get('seed_runs', []):
            for entry in run.get('epoch_history', []):
                metric_names.update(entry.get('val_metrics', {}).keys())
                metric_names.update(entry.get('test_metrics', {}).keys())
        return sorted(metric_names)

    def _fmt_bool(v):
        if v is True:
            return '[TRUE]'
        if v is False:
            return '[FALSE]'
        return '[NONE]'

    def _fmt_text(v):
        if v is None:
            return '[NONE]'
        s = str(v).upper()
        return f'[{s}]'

    def get_config_label(json_path):
        import json
        with open(json_path, 'r') as f:
            log_data = json.load(f)
        run_params = log_data.get('run_params', {})
        return (
            f"PB:{_fmt_bool(run_params.get('process_bridge'))}  "
            f"BS:{_fmt_text(run_params.get('bridge_strategy'))}  "
            f"PH:{_fmt_bool(run_params.get('process_hub'))}  "
            f"HS:{_fmt_text(run_params.get('hub_strategy'))}"
        )

    candidates = []
    for p1 in base.iterdir():
        if not p1.is_dir():
            continue
        if has_cfg_json(p1):
            candidates.append(p1)
        for p2 in p1.iterdir():
            if p2.is_dir() and has_cfg_json(p2):
                candidates.append(p2)

    sorted_candidates = sorted(candidates, key=lambda p: str(p))
    folder_options = [(p.name, str(p)) for p in sorted_candidates]
    if not folder_options:
        print(f'No folders with cfg_*.json found in: {base}')
        return None

    folder_dropdown = widgets.Dropdown(
        options=folder_options,
        value=folder_options[0][1],
        description='Folder:',
        layout=widgets.Layout(width='95%')
    )

    config_dropdown = widgets.Dropdown(
        options=[],
        description='Config:',
        layout=widgets.Layout(width='95%')
    )

    metric_dropdown = widgets.Dropdown(
        options=[],
        description='Metric:'
    )

    png_name_input = widgets.Text(
        value=f"{get_dataset_task_name(folder_options[0][1])}.png",
        description='PNG file:',
        layout=widgets.Layout(width='60%')
    )

    button = widgets.Button(description='Plot Selected', button_style='primary')
    save_button = widgets.Button(description='Save PNG', button_style='success')
    output = widgets.Output()
    last_fig = None

    def refresh_configs(*_):
        selected_folder = folder_dropdown.value
        configs = sorted_cfg_files(selected_folder)
        config_options = []
        for cfg in configs:
            json_path = str(Path(selected_folder) / cfg)
            label = f"{cfg} | {get_config_label(json_path)}"
            config_options.append((label, cfg))
        config_dropdown.options = config_options
        if config_options:
            config_dropdown.value = config_options[0][1]
        refresh_metrics()

    def refresh_metrics(*_):
        selected_folder = folder_dropdown.value
        selected_config = config_dropdown.value
        if not selected_config:
            metric_dropdown.options = []
            return
        json_path = str(Path(selected_folder) / selected_config)
        metrics = get_metric_names_for_config(json_path)
        metrics = prioritize_roc_auc(metrics)
        metric_dropdown.options = metrics
        if metrics:
            metric_dropdown.value = metrics[0]

    def update_default_png_name(*_):
        png_name_input.value = f"{get_dataset_task_name(folder_dropdown.value)}.png"

    def on_click(_):
        nonlocal last_fig
        with output:
            clear_output(wait=True)
            selected_folder = folder_dropdown.value
            selected_config = config_dropdown.value
            selected_metric = metric_dropdown.value
            json_path = str(Path(selected_folder) / selected_config)
            print(f'Using: {json_path}')
            print(f'Plotting train_loss with validation/test metric: {selected_metric}')
            last_fig = plot_single_config_val_test_metric(json_path, selected_metric)

    def on_save_click(_):
        nonlocal last_fig
        with output:
            import matplotlib.pyplot as plt

            fig_to_save = last_fig
            if fig_to_save is None:
                figure_numbers = plt.get_fignums()
                if figure_numbers:
                    fig_to_save = plt.figure(figure_numbers[-1])

            if fig_to_save is None:
                print('No figure to save yet. Click Plot Selected first.')
                return

            filename = png_name_input.value.strip() or f"{get_dataset_task_name(folder_dropdown.value)}.png"
            filename = Path(filename).name
            if not filename.lower().endswith('.png'):
                filename += '.png'

            plots_dir = get_plots_dir()
            plots_dir.mkdir(parents=True, exist_ok=True)
            save_path = plots_dir / filename
            fig_to_save.savefig(save_path, dpi=300, bbox_inches='tight')
            last_fig = fig_to_save
            print(f'Saved PNG: {save_path}')

    folder_dropdown.observe(refresh_configs, names='value')
    folder_dropdown.observe(update_default_png_name, names='value')
    config_dropdown.observe(refresh_metrics, names='value')
    refresh_configs()

    button.on_click(on_click)
    save_button.on_click(on_save_click)
    display(widgets.VBox([
        folder_dropdown,
        widgets.HBox([config_dropdown, metric_dropdown]),
        widgets.HBox([button, save_button]),
        png_name_input,
        output
    ]))
    return folder_dropdown, config_dropdown, metric_dropdown

choose_single_config_metric_plot()

(Dropdown(description='Folder:', layout=Layout(width='95%'), options=(('ctu-adventureworks_adventureworks-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-adventureworks_adventureworks-original/ctu-adventureworks_adventureworks-original_sage_edge_attr'), ('ctu-geneea_geneea-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-geneea_geneea-original/ctu-geneea_geneea-original_sage_edge_attr'), ('ctu-lahman_lahman-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-lahman_lahman-original/ctu-lahman_lahman-original_sage_edge_attr'), ('ctu-northwind_northwind-original_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/ctu-northwind_northwind-original/ctu-northwind_northwind-original_sage_edge_attr'), ('rel-amazon_item-churn_sage_edge_attr', '/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/rel-amazon_item-churn/rel-amazon_item-churn_sage_edge_attr'), ('rel-amazon_item-ltv_sage_edge_attr', '/hom